### Model Validation Report
# To do
| Field | Entry |
|---|---|
| Model name| Portfolio Value at Risk (VaR) Estimation: GARCH-Monte Carlo Simulation for EUR/USD|
| Model developer| Nikhil Srivastava | |
| Model validator| Joshua Sanchez |
| Model version | Version 1 |
| Dates for Model Creation | September 23, 2026 - October 8, 2026|
| Date of validation| October 29, 2026 - Date Validation ends |
| Overall conclusion | Approved / Approved with conditions / Not approved |

#### *Executive Summary*
# To do
(two or three sentences: what the model does, the overall conclusion, and the most important findings)
This Value at Risk (VaR) model assesses...
Based on this model we can conclude...
The most important findings are ... expected shortfall, 99% VaR

| Severity | Number of findings |
|---|---|
| High | |
| Medium |1|
| Low | 4|


# *i. Scope of Review*

(list the materials reviewed — documentation, code, data — and state anything you deliberately did not test, and why)

Materials reviewed: Version 1 of the EUR/USD GARCH Monte Carlo Value at Risk model, consisting of the developer’s documentation (purpose, scope, model outline, assumptions, limitations, and monitoring plan) and the accompanying Python notebook containing the code. The data are daily EUR/USD closing prices downloaded from Yahoo Finance using the yfinance package, with a requested window from September 21, 2021 to September 21, 2026. This data contains 1,299 daily closing prices.

Not tested: Alternative GARCH specifications (for example, GJR GARCH or EGARCH) were not tested as replacements for the developer’s model. GARCH(1,1) is a widely used baseline for VaR, and the purpose of this review is to assess the developer’s chosen specification against its stated scope, not to select a different model.

# To Do
maybe use another source for data (https://www.federalreserve.gov/datadownload/Build.aspx?rel=H10)
build another model
stress test the model with new data

# *ii. Purpose, Business Context and Intended Use*
# Review
*Mirrors developer sections i–iii*

(is the stated business purpose clear? Is the model's intended use appropriate, and does the developer state what it should NOT be used for? Is anything used outside its scope?)

### Developer Section
#### Purpose
Value at Risk (VaR) is a standard risk-management metric used across modern banking institutions to quantify the potential financial loss of an asset or portfolio over a specified time horizon for a given confidence level. Foreign exchange (FX) markets are characterized by time-varying volatility, dynamic variance clustering, and heavy-tailed yield distributions, making traditional static variance assumptions (such as standard normal distribution models) inadequate.

This project develops an advanced market risk measurement framework for a **EUR/USD foreign exchange portfolio**. The framework combines a **Generalized Autoregressive Conditional Heteroskedasticity (GARCH(1,1))** model to capture time-varying volatility dynamics with a **Monte Carlo Simulation** methodology to project potential portfolio loss distributions over a specified horizon. Using historical EUR/USD daily closing price data from **2021 through 2026**, this project evaluates market exposure to support daily risk management and capital reserve requirements.

#### Business Context
For a bank trading desk holding currency positions in EUR/USD, sharp movements driven by macroeconomic developments (e.g., ECB/Federal Reserve rate policy shifts, geopolitical shocks) create significant market risk. Implementing a hybrid dynamic VaR framework using **GARCH + Monte Carlo** solves this by:
* Dynamically updating risk estimates based on current market volatility regimes.
* Simulating thousands of forward return paths to capture complex tail outcomes accurately.
* Supporting compliance with Basel Committee market risk standards (e.g., Fundamental Review of the Trading Book - FRTB)

#### Intended Use
* **Underlying Asset Class:** Foreign Exchange (FX).
* **Target Currency Pair:** EUR/USD (spot rates).
* **Data Horizon:** Daily closing prices from **January 2021 to September 2026**.
* **Risk Metric:** Value at Risk (VaR) at **95%** and **99%** confidence levels.
* **Holding Horizon:** 1-day forward (scalable to 10-day using standard square-root time rules or multi-step simulation).
* **Out of Scope:** Multi-asset cross-currency portfolios, interest rate differentials (carry trade risk), options/derivatives pricing models, and intra-day high-frequency trading dynamics.


# *iii. Data Review*
*Mirrors developer data section*

Data is from Yahoo Finance using the 'yfinance' package, tracking daily EUR/USD closing prices (ticker EURUSD=X) using unadjusted closes. The window is from September 21, 2021 to September 21, 2026. This data contains 1,299 daily closing prices.
Train/Test Split follows: 
Train: 1038 obs  (2021-09-22 -> 2025-09-17)
Test : 260 obs  (2025-09-18 -> 2026-09-18)

Validator data checks found no missing closes, duplicate dates, non positive prices, or weekend rows. The largest absolute return (2.75% on April 11, 2025) agrees with the developer’s reported maximum. The only scaling is multiplication by a fixed constant (100) to express returns in percent, which is not fitted to data, and there is no missing value filling or capping. No leakage was found; the only input is the close price, converted to returns, so no feature depends on future information.

# *iv. Conceptual Soundness*
### Developer Section
## iv. Model Outline and Assumptions

### 1. Data Transformation (Log Returns)
Given daily spot price sequence $P_t$, simple returns are transformed into logarithmic returns $r_t$ to enforce time-additivity:

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

### 2. GARCH(1,1) Volatility Model
To capture volatility clustering, the dynamic conditional variance $\sigma_t^2$ is modeled as:

$$r_t = \mu + \epsilon_t, \quad \text{where } \epsilon_t = \sigma_t z_t \quad \text{and } z_t \sim \text{i.i.d. } \mathcal{N}(0, 1) \text{ or } t(\nu)$$

$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

Where:
* $\omega > 0$: Baseline constant variance.
* $\alpha \ge 0$: Reaction to recent market shocks (ARCH term).
* $\beta \ge 0$: Persistence of past volatility (GARCH term).
* Stationarity constraint: $\alpha + \beta < 1$.

### 3. Monte Carlo Simulation Engine
1. Estimate $\hat{\mu}, \hat{\omega}, \hat{\alpha}, \hat{\beta}$ from historical EUR/USD returns (2021–2026).
2. Extract the current spot price $P_T$ and the predicted baseline conditional volatility $\sigma_{T+1}$.
3. Draw $N$ standard random shocks $z_i \sim \mathcal{D}(0,1)$ (e.g., $N = 100,000$ iterations).
4. Simulate forward 1-day returns:

$$r_{T+1, i} = \hat{\mu} + \sigma_{T+1} z_i$$

5. Compute simulated portfolio value outcomes $V_{T+1, i} = V_T \cdot \exp(r_{T+1, i})$.
6. Value at Risk (VaR) at confidence level $1 - \alpha$ is the $\alpha$-quantile of simulated losses:

$$\text{VaR}_{\alpha} = V_T - \text{Percentile}\left(\{V_{T+1, i}\}_{i=1}^N, \alpha \times 100\right)$$

### Model Assumptions
1. **Stationarity:** The underlying process driving conditional parameters remains stationary over the calibration window.
2. **Distributional Structure:** Innovations $z_t$ are assume to follow a specified distribution (Standard Normal or Student’s $t$-distribution for heavy tails).
3. **No Liquidity Constraints:** Positions can be liquidated instantly at the prevailing market closing price without market impact.


| Assumption | How tested | Result | Holds? |
|---|---|---|---|
| Stationarity of returns | ADF and KPSS on returns | ADF statistic of negative 12.94 (p below 0.001), KPSS p-value at or above 0.10 | Yes |
| Alpha plus beta below 1 | GARCH fit |volatility persistence parameter of  0.995, very close to 1| Yes|
|Distributional Structure | Jarque-Bera and BIC|Normal rejected (p below 0.001), BIC 1,432 (t) vs 1,470 (Normal)|Yes for Student t|
|ARCH Effects|Ljung-Box on squared returns, ARCH-LM|both tests give p below 0.001, so constant variance is rejected, and volatility clustering is present|Yes|
|Constant Mean Equation is sufficient|Ljung-Box on returns|Ljung Box p value for returns is 0.0526, just above 0.05, and the estimated mean is near zero.|Yes|


Method appropriateness: GARCH(1,1) with Student t shocks and Monte Carlo simulation is appropriate for a single factor, linear EUR/USD spot position at a 1-day horizon. Volatility clustering is confirmed in the data (ARCH LM and Ljung-Box on squared returns both reject constant variance at p below 0.001), and the Normal assumption is rejected (Jarque-Bera p-value below 0.001, excess kurtosis of 1.97).

Alternatives: The notebook benchmarks EWMA and constant volatility and cross-checks against historical simulation, but this is not reflected in the documentation. The rejection of Normal innovations is well supported (BIC of 1,432 for Student t versus 1,470 for Normal). The documentation does not mention alternatives such as EWMA, constant volatility, or asymmetric GARCH variants, nor give reasons for rejecting them.

# *v. Replication*
*Mirrors developer section v*

(rerun the model using only the GitHub repo and the documentation, without help from the developer. Record any step that could not be reproduced)

| Reported metric | Developer's value | Replicated value | Match? |
|---|---|---|---|
| Price rows |  |1,299 | |
| Train and test observations | | 1,038 and 260 | |
| Selected distribution, nu || Student t, 6.49 | | |
| BIC,Student t vs Normal || 1,432.05 vs 1,469.99 | | |
| Alpha, beta (training fit) || 0.0296, 0.9654 | | |
| Next day volatility forecast || 0.3173% | | |
| 1 day 99% VaR, Monte Carlo || $0.8206 | | |
| 1 day 99% expected shortfall || $1.0262 | | | ## Why is this not creating a new row
| 10 day 99% VaR, full path || $2.556 | | |
| 1% breaches, GARCH t (Kupiec p) || 0 (0.0222) | | |
| 5% breaches, GARCH t (Kupiec p) || 8 (0.1267) | | |

The model was replicated from the GitHub repository and the documentation without help from the developer. All code ran without errors, and every reported metric above was reproduced. Steps that were underspecified: package installation commands are commented out with no stated versions. 

In [2]:
# Replication: rerun the developer's code and recompute their reported numbers

# *vi. Independent Testing*

(your own out-of-sample tests, using metrics suited to the model type, e.g.)
#### Can sample tests run from September 21 to today as a forward test, and do before September 2021, like March 2020-September 2021 as a stress test.
#### Can fit on a rolling window, like quarterly or monthly and test to see how it works
#### Used the Student t quantiles to test all of the benchmarks
- *VaR:* exception count, Basel traffic light, Kupiec and Christoffersen tests, backtest window including a stress period

(also include stability (cross-validation or rolling windows) and sensitivity to key choices such as the threshold or lookback window)

In [3]:
# Independent tests

# *vii. Benchmark / Challenger Model*
# To do
(build one alternative model and compare it with the developer's model on the same data and metrics. Explain any meaningful gap)

| Metric | Developer model | Challenger model |
|---|---|---|
| | | |

In [4]:
# Challenger model

# *viii. Limitations Review*
*Mirrors developer section vi (limitations)*

### Developer Section
### Model Limitations
1. **Model Parameter Sensitivity:** GARCH parameters are highly sensitive to extreme regime shifts (e.g., sudden financial crises or unexpected central bank rate decisions).
2. **Distributional Misspecification:** If standard normal distributions are assumed for innovations $z_t$, extreme tail events may still be underestimated compared to Student's $t$ or skewed distributions.
3. **Single Asset Focus:** Does not account for dynamic correlations with other currency pairs or cross-asset feedback loops.
(are the developer's stated limitations complete? List any undisclosed limitations you found)

# *ix. Ongoing Monitoring Review*
*Mirrors developer section vi (monitoring)*
### Developer Section
### Ongoing Monitoring & Governance
* **Daily Backtesting (Kupiec POF Test):** Compare actual EOD daily portfolio returns against predicted VaR levels. If the number of exceptions exceeds regulatory limits (e.g., >2.5% for 97.5% VaR), trigger model re-calibration.
* **Stress Testing & Scenario Analysis:** Supplement VaR metrics with historical stress tests (e.g., 2008 global financial crisis shock, post-pandemic inflation spikes) to account for tail-risk breaks.
* **Periodic Parameter Re-estimation:** Re-fit GARCH parameters monthly or quarterly to ensure parameters adapt to changing volatility regimes.

(are the proposed monitoring metrics appropriate, and do they have clear trigger levels that would prompt a review?)

#### *x. Findings*

| Rating | Meaning | Example |
|---|---|---|
| High | Results cannot be trusted or reproduced | Code does not run; data leakage |
| Medium | Usable, but a real weakness | Untested assumption; documentation does not match code |
| Low | Documentation or presentation issue | Wrong date; unclear wording |

| ID | Section | Finding | Evidence (cell, number or test) | Rating | Required action |
|---|---|---|---|---|---|
| F1 | Section iii of the Model Documentation|Section iii of the Model Documentation states, "Data Horizon: Daily closing prices from January 2021 to September 2026," but the data was actually collected from September 2021 to September 2026 | Cell [3]| Low|Correct Model documentation to reflect dates of data collection |
| F2 | | | | | |
### Findings
1. Low: Inconsistency in stating dates for data collection. Section iii of the Model Documentation states, "Data Horizon: Daily closing prices from January 2021 to September 2026," but the data was actually collected from September 2021 to September 2026.  Source: Cell [3]
2. Medium: A 95% VaR was listed as an intended risk metric, but was only used in the backtesting, not in the Monte Carlo simulation. Source: Model Documentation, Cells [34,35]
3. Low: In the Ongoing Monitoring & Governance section, "Re-fit GARCH parameters monthly or quarterly to ensure parameters adapt to changing volatility regimes." The backtest freezes parameters at their training values for the 260-day test window, which differs from the monitoring plan. 
4. Low: The developer applied Normal quantiles to the benchmark models and Student t quantiles to GARCH. Rerunning all three with the Student t quantile changed one 5% EWMA breach count (9 to 10) and no pass or fail result. Action: use a common quantile in the benchmark comparison.
5. Low: package installation commands are commented out with no stated versions

#### *xi. Overall Conclusion*

(Approved / Approved with conditions / Not approved — with a short justification. If conditional, list the findings that must be resolved first)

#### *xii. Developer Response*
*Completed by the model developer*

| ID | Agree / Dispute | Response or action taken | Date |
|---|---|---|---|
| F1 | | | |
| F2 | | | |

**Validator sign-off:** (for each finding, state whether it is closed, accepted with conditions, or still open)